# Entrenamiento y selección del mejor modelo

In [4]:
import torch
import torchvision.models as models
from torch import nn
from src.data import get_loaders
from src import CANONICAL_LABELS
import os

In [5]:
# --- Data ---
DATASET_ROOT = os.path.abspath(os.path.join("../data"))
train_loader, val_loader, test_loader = get_loaders(DATASET_ROOT, batch_size=32)

In [ ]:
# --- Modelo ---
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
model.classifier = nn.Linear(model.classifier.in_features, len(CANONICAL_LABELS))
nn.init.xavier_uniform_(model.classifier.weight)

In [ ]:
# --- Entrenamiento ---
def train_fine_tuning(model, learning_rate, num_epochs=5, param_group=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loss_fn = nn.BCEWithLogitsLoss()

    if param_group:
        params_1x = [p for n, p in model.named_parameters()
                     if n not in ["classifier.weight", "classifier.bias"]]
        optimizer = torch.optim.SGD([
            {'params': params_1x},
            {'params': model.classifier.parameters(), 'lr': learning_rate * 10}
        ], lr=learning_rate, weight_decay=0.001)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate,
                                    weight_decay=0.001)

    model.to(device)
    for epoch in range(num_epochs):
        model.train()
        L = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            l = loss_fn(model(X), y)
            l.backward()
            optimizer.step()
            L += l.item()
        print(f'epoch {epoch + 1}, loss {L / len(train_loader):.4f}')

train_fine_tuning(model, 5e-5)

# --- Guardar checkpoint ---
torch.save(model.state_dict(), "densenet121_finetuned.pth")